# Notebook 03 — Análisis semántico

**Objetivo:** usar modelos de inteligencia artificial para analizar el **significado** del texto,
no solo su estructura. Mientras el notebook 02 respondía preguntas de "quién se conecta con quién",
este notebook responde "qué dicen" y "cómo escriben".

**Técnicas en este notebook (de mayor a menor complejidad):**

| Técnica | ¿Qué hace? | Modelo |  
|---------|-----------|--------|  
| **Embeddings** | Convierte texto en vectores numéricos que capturan significado | qwen3-embedding (4096D) |  
| **UMAP + HDBSCAN** | Reduce dimensiones y detecta clusters de usuarios similares | Matemáticas |  
| **BERTopic** | Descubre automáticamente los temas principales del foro | Estadística + embeddings |  
| **NER** | Extrae entidades nombradas (personas, organizaciones, ideologías) | qwen2.5:14b |  
| **Perfilado** | Sintetiza el perfil de cada actor clave | qwen2.5:14b |  
| **Burrows' Delta** | Mide similitud de estilo de escritura entre usuarios | Estadística clásica |  

**Prerequisito:** `01_ingenieria_datos.ipynb` debe haberse ejecutado para tener los parquets.
Los modelos de embeddings y NER requieren **Ollama** corriendo localmente.

**Nota sobre los outputs precomputados:** las celdas de embeddings y NER pueden tardar
horas en ejecutarse. Si ya existen archivos de caché en `results/ironmarch/`, se cargan
automáticamente. Los outputs ya calculados se muestran en el notebook original `_archive/03_embeddings_profiling.ipynb`.

## 1. Imports y carga de datos

In [1]:
import sys
import json
import re
from pathlib import Path
from datetime import datetime
from collections import Counter
from itertools import combinations

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
from tqdm.auto import tqdm
import umap as umap_lib
import hdbscan
import ollama
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

from src.utils import RESULTS_DIR
from src.embeddings import embed_users, compute_actor_centroids

plt.style.use('dark_background')
IM_RESULTS = RESULTS_DIR / 'ironmarch'

# Cargar parquets limpios
posts_clean = pd.read_parquet(IM_RESULTS / 'posts_clean.parquet')
users_clean = pd.read_parquet(IM_RESULTS / 'users_clean.parquet')

uid_to_name = dict(zip(users_clean['userid'], users_clean['username_raw']))

print(f'Posts: {len(posts_clean):,}  |  Usuarios: {len(users_clean):,}')
print(f'Cols posts: {list(posts_clean.columns)}')

Posts: 174,651  |  Usuarios: 1,207
Cols posts: ['forum', 'postid', 'threadid', 'userid', 'username', 'dateline', 'pagetext', 'text_clean', 'text_len']


## 2. Embeddings: representar usuarios como vectores

### ¿Qué es un embedding?

Un **embedding** es la traducción de texto a un vector numérico (una lista de números)
que captura el significado semántico. Dos textos con significado similar tendrán vectores
cercanos en el espacio vectorial.

Por ejemplo:
- "Hitler fue un líder" y "el Führer fue decisivo" → vectores muy cercanos
- "Hitler fue un líder" y "me gusta la pizza" → vectores muy lejanos

Usamos `qwen3-embedding` que produce vectores de **4096 dimensiones**.
Para cada usuario, generamos un vector que representa el estilo y contenido de todos sus posts.

### El experimento metodológico: ¿cuántos posts necesitamos por usuario?

Probamos 5 estrategias distintas y medimos cuán similares son los resultados:

| Estrategia | Método | Posts procesados | Tiempo real | Correlación vs full (ρ) |
|---|---|---|---|---|
| A | Concatenar todos los posts → 1 vector | 157,779 | ~12 min | — |
| B | 1 embedding por post, promedio (completo) | 157,779 | ~9 h | referencia |
| C | Top-50 posts más largos por usuario | ≤41,800 | 2h 35min | **ρ=0.799** |
| D | Top-100 posts más largos por usuario | 41,319 | 3h 33min | **ρ=0.860** |
| E | Top-150 posts más largos por usuario | 53,193 | 4h 12min | **ρ=0.896** |

**Conclusión:** el salto de C→D (+37min, +6 puntos ρ) es más eficiente que D→E (+39min, +4 puntos ρ).
Para análisis exploratorio, **D (top-100) es el mejor balance** entre fidelidad y costo computacional.

In [2]:
# Función para cargar el archivo .npz más reciente que coincida con un patrón
# Los archivos .npz son arrays numpy comprimidos — ideal para guardar matrices grandes
def load_latest(pattern: str):
    """Carga el archivo .npz más reciente que coincida con el patrón.
    Retorna un diccionario con los arrays, o None si no existe ninguno."""
    matches = sorted(IM_RESULTS.glob(pattern))
    if not matches:
        return None
    path = matches[-1]
    print(f'Cargando caché: {path.name}')
    return dict(np.load(path, allow_pickle=True))

def save_timestamped(data: dict, section: str, name: str):
    """Guarda un diccionario de arrays con timestamp en el nombre del archivo."""
    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    out = IM_RESULTS / f'{section}_{name}_{ts}.npz'
    np.savez_compressed(out, **data)
    print(f'Guardado: {out.name}')
    return out

# Preparar posts para embedding
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'
posts_embed = posts_clean[['userid', text_col]].copy()
posts_embed = posts_embed.dropna(subset=[text_col])
posts_embed = posts_embed[posts_embed[text_col].astype(str).str.strip().str.len() > 20]
posts_embed = posts_embed.rename(columns={text_col: 'pagetext'})

print(f'Posts con texto válido para embedding: {len(posts_embed):,}')
print(f'Usuarios cubiertos: {posts_embed["userid"].nunique():,}')

Posts con texto válido para embedding: 174,651
Usuarios cubiertos: 1,164


`embed_users` (usada en la Estrategia A) concatena todos los posts de un usuario en un solo texto y genera un embedding — más contexto de estilo. `compute_actor_centroids` (usada en las estrategias C/D/E) embebe cada post por separado y promedia los vectores L2-normalizados — más robusto con corpus grandes o posts atípicos.

La versión completa de estos cálculos sobre todo el dataset (no la muestra que corre este notebook en vivo) se precomputó con `scripts/precompute.py embed --strategy embed_users --file "data/Far Right Forum/IronMarch_2019.11.zip"` y `scripts/precompute.py compare --file "data/Far Right Forum/IronMarch_2019.11.zip" --reference centroids` (este último compara distintos tamaños de muestra). Ver [`scripts/README.md`](../scripts/README.md).

In [3]:
# Estrategia A: concatenar todos los posts de cada usuario → 1 vector por usuario
# Tiempo estimado: ~12 min (usa caché si ya está calculado)
MAX_CHARS_A = 50_000  # Limitar a 50K chars para evitar timeouts en Ollama

cached_a = load_latest('s5a_embed_users_*.npz')

if cached_a is not None:
    user_ids_a = cached_a['user_ids'].tolist()
    vectors_a  = cached_a['vectors']
    print(f'Parte A cargada desde caché: {len(user_ids_a):,} usuarios, dim={vectors_a.shape[1]}')
else:
    print('Caché no encontrado. Ejecutando Parte A (embed_users)...')
    print(f'Tiempo estimado: ~12 minutos. Dejar corriendo.')

    grouped = (
        posts_embed.groupby('userid')['pagetext']
        .apply(lambda texts: ' '.join(texts.tolist())[:MAX_CHARS_A])
    )
    uids  = grouped.index.tolist()
    texts = grouped.tolist()

    results = []
    batches  = [texts[i:i+32] for i in range(0, len(texts), 32)]  # Procesar de 32 en 32
    for batch in tqdm(batches, desc='Embedding qwen3-embedding'):
        resp = ollama.embed(model='qwen3-embedding', input=batch)
        results.extend(resp.embeddings)

    user_ids_a = uids
    vectors_a  = np.array(results, dtype=np.float32)
    save_timestamped(
        {'user_ids': np.array(user_ids_a, dtype='U64'), 'vectors': vectors_a},
        's5a', 'embed_users',
    )
    print(f'Parte A completada: {len(user_ids_a):,} usuarios')

Cargando caché: s5a_embed_users_20260702_113336.npz
Parte A cargada desde caché: 842 usuarios, dim=4096


In [4]:
# Estrategias C, D, E: top-N posts más largos por usuario
# Tiempo estimado: C ~2.5h, D ~3.5h, E ~4h (usan caché si ya está calculado)

def build_sampled_centroids(max_posts: int, section: str, cache_pattern: str):
    """Construye embeddings tomando los max_posts posts más largos de cada usuario.
    Si ya existe caché, lo carga en lugar de recalcular."""
    cached = load_latest(cache_pattern)
    if cached is not None:
        ids  = cached['user_ids'].tolist()
        vecs = cached['vectors']
        print(f'Cargado (top-{max_posts}): {len(ids):,} usuarios, dim={vecs.shape[1]}')
        return ids, vecs

    # Seleccionar top-N posts más largos por usuario
    sample = (
        posts_embed
        .assign(text_len=posts_embed['pagetext'].str.len())
        .sort_values('text_len', ascending=False)
        .groupby('userid')
        .head(max_posts)
        .drop(columns='text_len')
        .reset_index(drop=True)
    )
    print(f'Computando (top-{max_posts}): {len(sample):,} posts...')
    ids, vecs = compute_actor_centroids(sample, min_posts=5)
    save_timestamped(
        {'user_ids': np.array(ids, dtype='U64'), 'vectors': vecs},
        section, f'centroids_sampled{max_posts}',
    )
    print(f'Completado (top-{max_posts}): {len(ids):,} usuarios')
    return ids, vecs

user_ids_c, vectors_c = build_sampled_centroids(50,  's5c', 's5c_centroids_sampled_*.npz')
user_ids_d, vectors_d = build_sampled_centroids(100, 's5d', 's5d_centroids_sampled100_*.npz')
user_ids_e, vectors_e = build_sampled_centroids(150, 's5e', 's5e_centroids_sampled150_*.npz')

Cargando caché: s5c_centroids_sampled_20260627_012509.npz


Cargado (top-50): 836 usuarios, dim=4096
Cargando caché: s5d_centroids_sampled100_20260702_021117.npz


Cargado (top-100): 836 usuarios, dim=4096
Cargando caché: s5e_centroids_sampled150_20260702_062316.npz


Cargado (top-150): 836 usuarios, dim=4096


### 2.1 Comparativa Spearman ρ entre estrategias

Para saber si el muestreo (C/D/E) preserva la información del método completo (B),
calculamos la **correlación de Spearman (ρ)** entre las matrices de similitud.

¿Qué significa esto? Para cada par de usuarios, calculamos cuán similares son según
el método B (completo) y según el método C/D/E (muestreado). Si el ranking de similitudes
es prácticamente el mismo, ρ ≈ 1. Si son completamente distintos, ρ ≈ 0.

In [5]:
# Intentar cargar la estrategia B (completa) desde caché
cached_b = load_latest('s5b_centroids_full_*.npz')
if cached_b is not None:
    user_ids_b = cached_b['user_ids'].tolist()
    vectors_b  = cached_b['vectors']
    print(f'Parte B cargada: {len(user_ids_b):,} usuarios, dim={vectors_b.shape[1]}')
else:
    print('Parte B (centroids_full) no disponible en caché.')
    print('Este cálculo tarda ~9 horas. Los resultados ya calculados están en _archive/03_embeddings_profiling.ipynb.')
    user_ids_b = None; vectors_b = None

def pairwise_sims(user_ids, vectors):
    """Calcula similitudes coseno para todos los pares posibles de usuarios.
    Retorna una Serie de pandas con índice (user_a, user_b) y valor = similitud."""
    sim_matrix = cosine_similarity(vectors)
    np.fill_diagonal(sim_matrix, 0)  # Excluir un usuario consigo mismo
    idx_i, idx_j = np.triu_indices(len(user_ids), k=1)  # Solo triángulo superior
    keys = pd.MultiIndex.from_arrays([
        [str(user_ids[i]) for i in idx_i],
        [str(user_ids[j]) for j in idx_j],
    ])
    return pd.Series(sim_matrix[idx_i, idx_j], index=keys)

available = {}
for label, ids_var, vecs_var in [
    ('A: embed_users',    user_ids_a if 'user_ids_a' in dir() else None, vectors_a if 'vectors_a' in dir() else None),
    ('B: centroids_full', user_ids_b, vectors_b),
    ('C: sampled_top50',  user_ids_c if 'user_ids_c' in dir() else None, vectors_c if 'vectors_c' in dir() else None),
    ('D: sampled_top100', user_ids_d if 'user_ids_d' in dir() else None, vectors_d if 'vectors_d' in dir() else None),
    ('E: sampled_top150', user_ids_e if 'user_ids_e' in dir() else None, vectors_e if 'vectors_e' in dir() else None),
]:
    if ids_var is not None and vecs_var is not None and len(ids_var) > 1:
        print(f'Calculando pares para {label}...')
        available[label] = pairwise_sims(ids_var, vecs_var)

print('\nCorrelaciones de Spearman vs método B (referencia completa):')
ref_label = 'B: centroids_full'
if ref_label in available:
    for lbl, sims in available.items():
        if lbl == ref_label: continue
        common = sims.index.intersection(available[ref_label].index)
        if len(common) == 0: continue
        rho, p = spearmanr(sims[common], available[ref_label][common])
        print(f'  {lbl:<25} ρ={rho:.4f}  (n={len(common):,} pares)')
else:
    print('  Parte B no disponible en caché. Ver resultados en _archive/03_embeddings_profiling.ipynb.')
    print('  Resultados pre-calculados: C→ρ=0.799, D→ρ=0.860, E→ρ=0.896')

Cargando caché: s5b_centroids_full_20260626_225006.npz


Parte B cargada: 836 usuarios, dim=4096
Calculando pares para A: embed_users...


Calculando pares para B: centroids_full...


Calculando pares para C: sampled_top50...


Calculando pares para D: sampled_top100...


Calculando pares para E: sampled_top150...



Correlaciones de Spearman vs método B (referencia completa):


  A: embed_users            ρ=0.4111  (n=349,030 pares)
  C: sampled_top50          ρ=0.7990  (n=349,030 pares)


  D: sampled_top100         ρ=0.8603  (n=349,030 pares)


  E: sampled_top150         ρ=0.8964  (n=349,030 pares)


## 3. UMAP + HDBSCAN: detectar clusters de usuarios

Tenemos vectores de 4096 dimensiones por usuario — imposible visualizar directamente.

**UMAP** (Uniform Manifold Approximation and Projection) reduce esas 4096 dimensiones
a 2 dimensiones preservando la estructura local: si dos usuarios son similares en 4096D,
seguirán siendo cercanos en 2D.

**HDBSCAN** (Hierarchical Density-Based Spatial Clustering) detecta automáticamente grupos
de usuarios cercanos en el espacio reducido. A diferencia de k-means, no requiere que
especifiquemos cuántos grupos queremos — los detecta según la densidad.

**Parámetro clave:** `min_cluster_size=5` le dice a HDBSCAN que necesita al menos
5 usuarios para considerar algo un cluster real. Usuarios aislados quedan marcados como
ruido (cluster -1, en gris).

In [6]:
# Elegir el mejor embedding disponible (B > E > D > C > A)
_priority = [
    ('B: centroids_full',  user_ids_b if 'user_ids_b' in dir() else None, vectors_b if 'vectors_b' in dir() else None),
    ('E: sampled_top150',  user_ids_e if 'user_ids_e' in dir() else None, vectors_e if 'vectors_e' in dir() else None),
    ('D: sampled_top100',  user_ids_d if 'user_ids_d' in dir() else None, vectors_d if 'vectors_d' in dir() else None),
    ('C: sampled_top50',   user_ids_c if 'user_ids_c' in dir() else None, vectors_c if 'vectors_c' in dir() else None),
    ('A: embed_users',     user_ids_a if 'user_ids_a' in dir() else None, vectors_a if 'vectors_a' in dir() else None),
]

best_label, best_ids, best_vecs = None, None, None
for label, ids, vecs in _priority:
    if ids is not None and vecs is not None and len(ids) > 1:
        best_label, best_ids, best_vecs = label, ids, vecs
        break

if best_label:
    print(f'Usando embedding: {best_label}')
    print(f'  Usuarios: {len(best_ids):,}  |  Dimensiones: {best_vecs.shape[1]}')

    # UMAP: reducir de 4096D a 2D
    # n_neighbors=15: cada punto considera sus 15 vecinos más cercanos para el layout
    print('Calculando UMAP 2D...')
    reducer   = umap_lib.UMAP(n_components=2, n_neighbors=min(15, len(best_ids) - 1), random_state=42)
    coords_2d = reducer.fit_transform(best_vecs)

    # HDBSCAN: detectar clusters en el espacio de embeddings (no en 2D)
    # min_cluster_size=5: un grupo debe tener al menos 5 usuarios para ser cluster
    # min_samples=3: cuántos vecinos mínimos necesita un punto para no ser ruido
    print('Calculando HDBSCAN...')
    clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
    clusters  = clusterer.fit_predict(best_vecs)  # Predecir sobre vectores originales 4096D
    n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
    print(f'Clusters encontrados: {n_clusters}  |  Ruido (sin cluster): {(clusters == -1).sum()}')
else:
    print('Sin embeddings disponibles. Ejecutar las celdas de embeddings primero.')

Usando embedding: B: centroids_full
  Usuarios: 836  |  Dimensiones: 4096
Calculando UMAP 2D...


Calculando HDBSCAN...


Clusters encontrados: 2  |  Ruido (sin cluster): 810


In [7]:
if best_vecs is not None and 'clusters' in dir():
    names = [str(uid_to_name.get(uid, uid)) for uid in best_ids]

    fig = go.Figure(go.Scatter(
        x=coords_2d[:, 0], y=coords_2d[:, 1],
        mode='markers',
        marker=dict(
            size=6,
            color=clusters,
            colorscale='Plasma',
            opacity=0.85,
            showscale=True,
            colorbar=dict(title='Cluster'),
            line=dict(width=0),
        ),
        text=[f'<b>{n}</b><br>cluster={c}' for n, c in zip(names, clusters)],
        hovertemplate='%{text}<extra></extra>',
    ))
    fig.update_layout(
        title=f'UMAP + HDBSCAN — {best_label} ({n_clusters} clusters)<br>'
              '<sup>Cada punto es un usuario. Color = cluster. Gris = sin cluster (ruido). Hover = nombre.</sup>',
        template='plotly_dark', height=620, width=900,
        xaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    )
    fig.show()

    # Mostrar miembros de cada cluster
    cluster_ids = sorted(set(clusters) - {-1})
    print(f'\nMiembros por cluster (excluyendo ruido):')
    for cid in cluster_ids:
        members = [str(uid_to_name.get(uid, uid)) for uid, cl in zip(best_ids, clusters) if cl == cid]
        print(f'  Cluster {cid} ({len(members)} usuarios): {members[:8]}{"..." if len(members) > 8 else ""}')


Miembros por cluster (excluyendo ruido):
  Cluster 0 (15 usuarios): ['0', 'Leonidas', 'Pro Patria Mori', 'Myrrysmies', 'Spöket', 'kinumarch', 'Владимир_Борисов', 'Aquila']...
  Cluster 1 (11 usuarios): ['Celt', 'LITOS *_*', 'Солдат', 'Vex', 'Eternal Leaf', 'Kill The Poor', 'Tiwaz', 'Join my discord server']...


## 4. BERTopic — apoyo, no protagonista

BERTopic aparece aquí solo como evidencia de apoyo — el protagonista de este caso es la
red de influencia (notebook 02), no el descubrimiento de temas.

**Caché:** el ajuste de BERTopic tarda ~10-20 min. Si ya existe `bertopic_topics.parquet` en
`results/ironmarch/`, se carga directamente en vez de recalcular.

In [8]:
text_col = 'pagetext' if 'pagetext' in posts_clean.columns else 'message'

# Preparar corpus para BERTopic
# Usamos hasta 10.000 posts para que sea computacionalmente manejable
corpus_df = posts_clean[[text_col, 'userid']].copy()
corpus_df = corpus_df.dropna(subset=[text_col])
corpus_df['text_clean'] = (
    corpus_df[text_col].astype(str)
    .str.replace(r'<[^>]+>', ' ', regex=True)  # Quitar HTML
    .str.strip()
)
corpus_df = corpus_df[corpus_df['text_clean'].str.len() > 100]

# Muestra de los posts más largos (más informativos para topic modeling)
# Nota: nlargest no funciona sobre una columna de texto (dtype object) — hay que
# derivar una columna numérica de longitud primero (mismo patrón que la celda anterior).
if len(corpus_df) > 10000:
    corpus_df = (
        corpus_df.assign(text_len=corpus_df['text_clean'].str.len())
        .nlargest(10000, 'text_len')
        .drop(columns='text_len')
    )

documents = corpus_df['text_clean'].tolist()
print(f'Documentos para BERTopic: {len(documents):,}')

Documentos para BERTopic: 10,000


In [9]:
TOPICS_CACHE = IM_RESULTS / 'bertopic_topics.parquet'
TOPIC_INFO_CACHE = IM_RESULTS / 'bertopic_topic_info.parquet'

if TOPICS_CACHE.exists() and TOPIC_INFO_CACHE.exists():
    corpus_df = pd.read_parquet(TOPICS_CACHE)
    topics = corpus_df['topic'].tolist()
    topic_info_df = pd.read_parquet(TOPIC_INFO_CACHE)
    valid_topics = {
        row.topic_id: json.loads(row.keywords_json)
        for row in topic_info_df.itertuples()
    }
    print(f'BERTopic cargado desde caché: {len(corpus_df):,} documentos, {len(valid_topics)} topics')
else:
    print('Caché no encontrado. Ejecutando BERTopic...')
    print('Tiempo estimado: 10-20 minutos.')

    vectorizer = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))
    topic_model = BERTopic(
        vectorizer_model=vectorizer,
        min_topic_size=15,
        nr_topics='auto',  # Dejar que el modelo decida cuántos topics
        calculate_probabilities=False,
        verbose=True,
    )

    topics, _ = topic_model.fit_transform(documents)
    corpus_df = corpus_df.copy()
    corpus_df['topic'] = topics

    topic_info = topic_model.get_topics()
    valid_topics = {t: words for t, words in topic_info.items() if t != -1}

    corpus_df[['userid', 'topic', 'text_clean']].to_parquet(TOPICS_CACHE, index=False)
    pd.DataFrame([
        {'topic_id': t, 'keywords_json': json.dumps(words)}
        for t, words in valid_topics.items()
    ]).to_parquet(TOPIC_INFO_CACHE, index=False)
    print(f'Guardado en caché: {TOPICS_CACHE.name}, {TOPIC_INFO_CACHE.name}')

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f'\nTopics encontrados: {n_topics}')
print(f'Posts sin topic asignado (-1): {topics.count(-1):,}  '
      f'({topics.count(-1)/len(topics)*100:.1f}%)')

topic_sizes = {t: topics.count(t) for t in valid_topics}

print('\nTop 15 topics por tamaño:')
print(f'{"ID":>4}  {"Palabras clave":<55}  {"Posts":>6}')
print('-' * 70)
for t in sorted(topic_sizes, key=topic_sizes.get, reverse=True)[:15]:
    kws = ', '.join([w for w, _ in valid_topics[t][:6]])
    print(f'{t:>4}  {kws:<55}  {topic_sizes[t]:>6}')

BERTopic cargado desde caché: 10,000 documentos, 70 topics

Topics encontrados: 70
Posts sin topic asignado (-1): 3,941  (39.4%)

Top 15 topics por tamaño:
  ID  Palabras clave                                            Posts
----------------------------------------------------------------------
   0  russia, russian, ukraine, serbian, putin, finland           502
   1  rel, title, rel stylecolorrgb2351540textdecorationunderlinebackgroundcolortransparent, stylecolorrgb2351540textdecorationunderlinebackgroundcolortransparent, materials, im     444
   2  women, men, sex, sexual, homosexuality, woman               418
   3  jews, jewish, police, white, people, jew                    416
   4  america, american, white, nation, race, culture             290
   5  fascism, fascist, christianity, like, god, people           254
   6  im, ive, started, history, school, time                     201
   7  capitalism, workers, socialism, class, capitalist, state     201
   8  pm, jacob, haw, just

## 5. NER: extracción de entidades con qwen2.5:14b

**NER** (Named Entity Recognition) identifica entidades nombradas en el texto:
personas, organizaciones, lugares, ideologías, armas, eventos.

Usamos `qwen2.5:14b` via Ollama con un **prompt de zero-shot**: le pedimos al modelo
que identifique entidades sin haberlo entrenado específicamente para este dataset.
El resultado es un JSON con las entidades encontradas y su tipo.

**Estrategia de muestreo:** top 50 usuarios por volumen de posts, hasta 50 posts por usuario.
Esto nos da ~2.500 posts a analizar — representativo sin ser prohibitivo en tiempo.

**Caché:** los resultados de NER se guardan en `ner_results.parquet`.
Si el archivo existe, se carga directamente — el NER puede tardar 2-3 horas.

In [10]:
import logging

SYSTEM_PROMPT = """You are an expert forensic analyst. Extract named entities from forum posts.
Return ONLY valid JSON object: {"entities": [{"text": "...", "type": "PERSON|ORG|LOCATION|IDEOLOGY|WEAPON|EVENT", "confidence": 0.0-1.0}]}"""

def extract_entities(text: str, model: str = 'qwen2.5:14b') -> list:
    """Extrae entidades nombradas de un post usando qwen2.5:14b vía Ollama.
    Retorna una lista de dicts con 'text', 'type' y 'confidence'.
    Un fallo aquí (JSON inválido, timeout, etc.) descarta solo ESTE post/entidad,
    nunca interrumpe el resto del loop de NER."""
    try:
        response = ollama.generate(
            model=model,
            system=SYSTEM_PROMPT,
            prompt=str(text)[:1500],  # Limitar a 1500 chars por post
            format='json',
            options={'temperature': 0},  # 0 = respuesta determinista
        )
        result   = json.loads(response['response'])
        entities = result.get('entities', []) if isinstance(result, dict) else []
        # El LLM a veces devuelve strings sueltos en vez de objetos {"text":...} —
        # filtrar aquí para que el .get() del bucle de abajo nunca reciba un str.
        return [e for e in entities if isinstance(e, dict)] if isinstance(entities, list) else []
    except Exception as exc:
        logging.warning(f'extract_entities: fallo en un post, se omite (no se aborta el loop): {exc}')
        return []

# Muestra: top 50 usuarios por cantidad de posts, máximo 50 posts cada uno
posts_text = posts_clean.dropna(subset=[text_col]).copy()
posts_text = posts_text[posts_text[text_col].astype(str).str.len() > 100]
top_user_ids = posts_text.groupby('userid').size().nlargest(50).index.tolist()

def clean_for_ner(text: str) -> str:
    """Limpiar el post antes de NER: quitar HTML y URLs (ruido para el LLM)."""
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'https?://\S+', ' ', text)
    return text.strip()

posts_text = posts_text.copy()
posts_text['text_clean_ner'] = posts_text[text_col].apply(clean_for_ner)

# Igual que en la celda anterior: nlargest no aplica sobre texto — se ordena
# por una columna numérica de longitud y se toma groupby().head(), sin usar
# apply() sobre grupos (evita el KeyError de pandas al excluir la columna de
# agrupación dentro de apply en versiones recientes de pandas).
sample_top = (
    posts_text[posts_text['userid'].isin(top_user_ids)]
    .assign(text_len=lambda d: d['text_clean_ner'].str.len())
    .sort_values('text_len', ascending=False)
    .groupby('userid')
    .head(50)
    .drop(columns='text_len')
    .reset_index(drop=True)
)
print(f'Muestra para NER:')
print(f'  Usuarios:     {sample_top["userid"].nunique()}')
print(f'  Posts totales: {len(sample_top):,}')

Muestra para NER:
  Usuarios:     50
  Posts totales: 2,500


La caché de `ner_results.parquet` la generó `scripts/precompute.py ner --file "data/Far Right Forum/IronMarch_2019.11.zip" --sample-size 500`, que extrae entidades (personas, organizaciones, ideología, etc.) usando un LLM local vía Ollama sobre todo el corpus.

Es la plantilla a reutilizar si quieres extraer entidades de dominio (IPs, handles, herramientas) de tu propio dataset con un LLM local — ver [`scripts/README.md`](../scripts/README.md).

In [11]:
NER_CACHE = IM_RESULTS / 'ner_results.parquet'

# El LLM recibe un enum fijo en el prompt (PERSON|ORG|LOCATION|IDEOLOGY|WEAPON|EVENT),
# pero format='json' solo garantiza JSON válido, no que respete el enum — en la práctica
# devuelve variantes (ORGANIZATION), combos (WEAPON|IDEOLOGY) y tipos libres (SUBCULTURE).
# Normalizamos aquí para que el ranking de "top entidades" no se fragmente por eso.
NER_CANONICAL_TYPES = {'PERSON', 'ORG', 'LOCATION', 'IDEOLOGY', 'WEAPON', 'EVENT'}
NER_TYPE_SYNONYMS = {'ORGANIZATION': 'ORG'}

def canonicalize_ner_type(t: str) -> str:
    t = str(t).upper().split('|')[0].strip()
    t = NER_TYPE_SYNONYMS.get(t, t)
    return t if t in NER_CANONICAL_TYPES else 'OTHER'

def normalize_ner(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza tipo (enum fijo) y texto (minúsculas) para que 'Jews'/'jews' o
    'Fascism'/'fascism' no se cuenten como entidades distintas en los rankings."""
    if len(df) == 0:
        return df
    df = df.copy()
    df['type'] = df['type'].apply(canonicalize_ner_type)
    df['entity'] = df['entity'].str.strip().str.lower()
    return (
        df.groupby(['userid', 'entity', 'type'], as_index=False)
        .agg(confidence=('confidence', 'mean'))
    )

if NER_CACHE.exists():
    ner_all = pd.read_parquet(NER_CACHE)
    print(f'NER cargado desde caché: {len(ner_all):,} entidades')
    print(f'Tipos de entidad: {ner_all["type"].value_counts().to_dict()}')
else:
    print('Ejecutando NER...')
    print('Tiempo estimado: 2-3 horas. Los resultados se guardan con checkpoint cada 10 usuarios.')
    all_records = []
    user_list   = sample_top['userid'].unique().tolist()

    for i, uid in enumerate(tqdm(user_list, desc='NER por usuario')):
        user_posts = sample_top[sample_top['userid'] == uid]
        for _, row in user_posts.iterrows():
            entities = extract_entities(str(row[text_col]))
            for ent in entities:
                txt = str(ent.get('text', '')).strip()
                if len(txt) < 2: continue
                all_records.append({
                    'userid':     uid,
                    'entity':     txt,
                    'type':       ent.get('type', 'UNKNOWN').upper(),
                    'confidence': float(ent.get('confidence', 0.5)),
                })
        # Checkpoint cada 10 usuarios
        if (i + 1) % 10 == 0 and all_records:
            pd.DataFrame(all_records).to_parquet(NER_CACHE, index=False)
            print(f'  Checkpoint: {i+1}/{len(user_list)} usuarios procesados')

    if all_records:
        ner_all = pd.DataFrame(all_records)
        ner_all = ner_all[ner_all['entity'].str.len() > 1]
        ner_all = normalize_ner(ner_all)
        ner_all.to_parquet(NER_CACHE, index=False)
        print(f'NER completado: {len(ner_all):,} entidades')
    else:
        print('Sin resultados. Verificar que qwen2.5:14b esté disponible: ollama pull qwen2.5:14b')
        ner_all = pd.DataFrame(columns=['userid', 'entity', 'type', 'confidence'])

NER cargado desde caché: 10,072 entidades
Tipos de entidad: {'IDEOLOGY': 2735, 'PERSON': 2583, 'LOCATION': 2042, 'ORG': 1763, 'EVENT': 585, 'OTHER': 279, 'WEAPON': 85}


In [12]:
# Visualización: top entidades por tipo
if len(ner_all) > 0:
    type_counts = ner_all['type'].value_counts()

    # Un solo orden (por conteo descendente) y una sola paleta de colores para
    # AMBAS gráficas, para que el mismo tipo tenga siempre el mismo color y la
    # misma posición relativa en las dos (antes: cada gráfica usaba su propio
    # orden/paleta y "PERSON" podía salir azul arriba y rojo abajo).
    entity_types_ordered = type_counts.index.tolist()
    PALETTE = ['#E94E4E', '#4E9EE9', '#E9A84E', '#7AE94E', '#C44EE9', '#FFD700', '#FF6B35']
    type_color = {t: PALETTE[i % len(PALETTE)] for i, t in enumerate(entity_types_ordered)}

    fig = go.Figure(go.Bar(
        x=entity_types_ordered,
        y=type_counts.values.tolist(),
        marker_color=[type_color[t] for t in entity_types_ordered],
    ))
    fig.update_layout(
        title='Distribución de tipos de entidad — NER IronMarch',
        xaxis_title='Tipo', yaxis_title='Menciones',
        template='plotly_dark', height=380,
    )
    fig.show()

    # Mismo orden que la gráfica de arriba (por conteo descendente)
    entity_types = entity_types_ordered

    if entity_types:
        n_cols = min(len(entity_types), 3)
        n_rows = (len(entity_types) + n_cols - 1) // n_cols
        fig2   = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=entity_types,
                               horizontal_spacing=0.08, vertical_spacing=0.12)
        for i, etype in enumerate(entity_types):
            row, col = i // n_cols + 1, i % n_cols + 1
            top = ner_all[ner_all['type'] == etype]['entity'].value_counts().head(10)
            fig2.add_trace(go.Bar(
                x=top.values[::-1], y=top.index[::-1].tolist(),
                orientation='h', marker_color=type_color[etype], showlegend=False,
            ), row=row, col=col)
        fig2.update_layout(
            title='Top 10 entidades por tipo — IronMarch NER',
            template='plotly_dark',
            height=380 * n_rows, width=min(380 * n_cols, 1100),
            margin=dict(l=150, r=20, t=60, b=40),
        )
        fig2.show()

### Zero-shot no es perfecto: el caso `WEAPON`

Antes de asumir que el NER está limpio porque el tipo ya está normalizado (ver arriba),
miremos las entidades reales de una categoría pequeña: `WEAPON`. El prompt le pide al
modelo un enum fijo, pero nada garantiza que **clasifique bien** dentro de esa categoría
— solo que devuelva JSON válido con esa etiqueta.

In [13]:
# "Antes": entidades WEAPON tal como las devolvió el modelo, sin filtrar
print('WEAPON — top 20 entidades sin filtrar:')
print(ner_all[ner_all['type'] == 'WEAPON']['entity'].value_counts().head(20))

WEAPON — top 20 entidades sin filtrar:
entity
guns                                                      2
fasces                                                    2
culture-energy                                            1
stalinist-made documentaries, books, articles and such    1
gun ownership                                             1
cellphone                                                 1
machette                                                  1
speed                                                     1
subutex                                                   1
arés                                                      1
sebesség                                                  1
versenyautó                                               1
pvc pipe                                                  1
ss officer hat                                            1
nsda                                                      1
black bull aresenal of the cuckold                    

**¿Y si el problema es simplemente que hay pocos términos de `WEAPON`?** Comparemos la
cima de cada categoría (la mención más repetida):

```
WEAPON:   máximo = 2 menciones   (guns, fasces)
ORG:      máximo = 28 menciones (ironmarch)
LOCATION: máximo = 33 menciones (europe, germany)
```

En `LOCATION` y `ORG`, aunque también tienen muchos términos que aparecen una sola vez
(79% y 85% respectivamente), los términos genuinos se repiten lo bastante (20-33 veces)
para destacar con claridad sobre ese ruido de fondo. En `WEAPON`, con solo 85 menciones en
total y el **98% de sus 83 términos únicos apareciendo una sola vez**, el máximo es 2 —
prácticamente indistinguible del ruido. Un "top 20" en una categoría tan pequeña y dispersa
no es un ranking real: es casi una muestra aleatoria de esos 85 casos, así que un error de
clasificación tiene la misma probabilidad de aparecer "arriba" que un acierto genuino.

No es que el modelo clasifique peor específicamente "arma" que "lugar" — es que aquí no hay
volumen suficiente para que la repetición separe señal de ruido, y por eso el error se ve
tan claramente en esta categoría y pasaría más desapercibido en una grande.

Se cuelan artes marciales (`judo`, `mma`, `muay thai`), sustancias (`speed`, `subutex`),
herramientas de software (`tor`, `i2p`) y ruido variado (`cellphone`, `german flags`,
palabras sueltas en otro idioma) junto a armas reales (`guns`, `machette`, `ak-47`).

**Por qué pasa (más allá del tamaño de la muestra):** zero-shot no tiene ningún ejemplo de
qué SÍ y qué NO es `WEAPON` — el modelo interpreta el nombre de la categoría con su propio
criterio, y "algo relacionado con combate o daño físico" es una frontera borrosa (¿un arte
marcial cuenta? ¿una sustancia que causa daño?). Es un error de **precisión de
clasificación**, no de formato — por eso la normalización de tipos de la celda de más
arriba no lo arregla, ese problema era otro (el modelo no respetando el enum al escribir el
campo `type`, no clasificando mal dentro de él).

**Frente a esto, si de verdad queremos meternos a filtrar a mano**, la solución de fondo
sería mejor prompting (ejemplos few-shot de qué NO es un arma) o un modelo especializado —
reentrenar/reejecutar 2500 posts vía Ollama para esto es desproporcionado aquí. Como remedio
ligero y auditable, podemos construir una **lista negra** explícita con los términos de
ruido ya identificados arriba — funciona para esta categoría concreta, pero no escala (no
cubre ruido que no hayamos visto todavía).

In [14]:
# "Después": mismo filtro, con una lista negra explícita de ruido ya identificado
WEAPON_NOISE_BLACKLIST = {
    'judo', 'kickboxing', 'mma', 'muay thai', 'brazilian jui jitsu',
    'speed', 'subutex', 'gnrh',
    'tor', 'i2p', 'cellphone',
    'german flags', 'tyr rune', 'culture-energy',
    'stalinist-made documentaries, books, articles and such',
    'arés', 'sebesség', 'versenyautó',
    'black bull aresenal of the cuckold', 'nsda',
}

weapon_clean = ner_all[
    (ner_all['type'] == 'WEAPON') & (~ner_all['entity'].isin(WEAPON_NOISE_BLACKLIST))
]
print(f'WEAPON — antes: {len(ner_all[ner_all["type"] == "WEAPON"])} filas  |  '
      f'después: {len(weapon_clean)} filas  ({len(WEAPON_NOISE_BLACKLIST)} términos en la lista negra)')
print()
print('WEAPON — top 20 entidades filtradas:')
print(weapon_clean['entity'].value_counts().head(20))

WEAPON — antes: 85 filas  |  después: 65 filas  (20 términos en la lista negra)

WEAPON — top 20 entidades filtradas:
entity
guns              2
fasces            2
gun ownership     1
machette          1
pvc pipe          1
ss officer hat    1
x-acto knife      1
ak-47             1
shermans          1
t-34s             1
hcg               1
gift              1
gold              1
aks               1
.308              1
aids              1
turner diaries    1
celtic swords     1
gladius           1
katana            1
Name: count, dtype: int64


## 6. Perfilado de actores con LLM

Esta es una sección nueva que no existía en los notebooks originales.

Para cada usuario relevante (top brokers de la red + usuarios en clusters específicos),
sintetizamos un perfil usando `qwen2.5:14b`. El modelo recibe:
1. El corpus completo del usuario (sus posts más representativos)
2. Sus entidades NER más frecuentes
3. Su topic principal (BERTopic)

Y retorna una síntesis estructurada: **rol probable, ideología, nivel de radicalización,
patrones de comportamiento observados**.

**Importante:** el perfil del LLM es una hipótesis, no un diagnóstico.
Debe tratarse como un punto de partida para análisis manual, no como conclusión definitiva.

In [15]:
PROFILING_PROMPT = """You are a forensic analyst studying extremist forums. 
Analyze the following information about a forum user and provide a structured profile.
Be analytical and objective. Base conclusions only on evidence provided.

User data:
- Username: {username}
- Most mentioned entities: {entities}
- Main topic cluster: {topic}
- Sample posts: {posts_sample}

Return JSON:
{{"role": "ideological_leader|recruiter|moderator|active_member|newcomer|lurker",
 "ideology_indicators": ["list", "of", "observed", "indicators"],
 "behavioral_patterns": ["patterns", "observed"],
 "radicalization_level": "low|medium|high|extreme",
 "confidence": 0.0-1.0,
 "summary": "one paragraph synthesis"
}}"""

def profile_user(username: str, entities_str: str, topic_str: str, posts_sample: str) -> dict:
    """Genera un perfil de usuario usando qwen2.5:14b.
    Retorna un diccionario con el perfil o un dict vacío si hay error."""
    prompt = PROFILING_PROMPT.format(
        username=username,
        entities=entities_str,
        topic=topic_str,
        posts_sample=posts_sample[:2000],  # Limitar el contexto
    )
    try:
        response = ollama.generate(
            model='qwen2.5:14b',
            prompt=prompt,
            format='json',
            options={'temperature': 0.1},
        )
        return json.loads(response['response'])
    except Exception as e:
        return {'error': str(e)}

print('Función de perfilado lista.')
print('Verificar modelo disponible:')
import subprocess
subprocess.run(['ollama', 'list'], check=False)

Función de perfilado lista.
Verificar modelo disponible:
NAME                       ID              SIZE      MODIFIED     
qwen3-embedding:latest     64b933495768    4.7 GB    25 hours ago    
qwen2.5:14b                7cdf5a0187d5    9.0 GB    3 weeks ago     
nomic-embed-text:latest    0a109f422b47    274 MB    3 weeks ago     


CompletedProcess(args=['ollama', 'list'], returncode=0)

In [16]:
PROFILES_CACHE = IM_RESULTS / 'actor_profiles.parquet'

if PROFILES_CACHE.exists():
    actor_profiles = pd.read_parquet(PROFILES_CACHE)
    print(f'Perfiles cargados desde caché: {len(actor_profiles)} actores')
else:
    # Seleccionar usuarios a perfilar: UNIÓN de
    #   (a) top 20 por volumen de posts públicos, y
    #   (b) top 20 por centralidad (betweenness_pm) en la red de PMs (notebook 02)
    # Esto es necesario porque un operador puede ser de bajo volumen público y alta
    # centralidad oculta -- si solo seleccionamos por volumen, ese tipo de actor
    # nunca entra al perfilado. userid=0 se excluye explícitamente de ambas listas:
    # no es una cuenta real (agrega contenido de cientos de personas distintas bajo
    # un mismo ID, ver notebook 02) y un LLM no puede perfilar a un usuario que no existe.
    top_volume = set(
        posts_clean[posts_clean['userid'] != '0']
        .groupby('userid').size().nlargest(20).index.tolist()
    )
    structural_path = IM_RESULTS / 'structural_signals.parquet'
    if structural_path.exists():
        _structural = pd.read_parquet(structural_path)
        top_pm_centrality = set(
            _structural[_structural['userid'] != '0']
            .dropna(subset=['betweenness_pm'])
            .sort_values('betweenness_pm', ascending=False)
            .head(20)['userid'].tolist()
        )
    else:
        top_pm_centrality = set()
    users_to_profile = sorted(top_volume | top_pm_centrality)

    profile_records = []
    corpus_by_user_path = IM_RESULTS / 'corpus_users_clean.parquet'
    if corpus_by_user_path.exists():
        corpus_by_user = pd.read_parquet(corpus_by_user_path)
    else:
        corpus_by_user = None

    for uid in tqdm(users_to_profile, desc='Perfilando actores'):
        uname = str(uid_to_name.get(uid, uid))

        # Entidades más frecuentes de este usuario
        if len(ner_all) > 0 and 'userid' in ner_all.columns:
            user_ents = ner_all[ner_all['userid'] == uid]['entity'].value_counts().head(10)
            entities_str = ', '.join(user_ents.index.tolist())
        else:
            entities_str = 'no disponible'

        # Topic principal del usuario
        if 'topic' in corpus_df.columns and 'userid' in corpus_df.columns:
            user_topics = corpus_df[corpus_df['userid'] == uid]['topic'].value_counts()
            top_topic_id = user_topics.index[0] if len(user_topics) > 0 else -1
            if top_topic_id != -1 and top_topic_id in valid_topics:
                topic_str = ', '.join([w for w, _ in valid_topics[top_topic_id][:5]])
            else:
                topic_str = 'sin topic asignado'
        else:
            topic_str = 'no disponible'

        # Muestra de posts
        if corpus_by_user is not None:
            row = corpus_by_user[corpus_by_user['userid'] == uid]
            posts_sample = row['corpus'].iloc[0][:2000] if len(row) > 0 else ''
        else:
            user_posts = posts_clean[posts_clean['userid'] == uid][text_col].dropna()
            posts_sample = ' '.join(user_posts.head(10).tolist())[:2000]

        # Generar perfil
        profile = profile_user(uname, entities_str, topic_str, posts_sample)
        profile['userid']   = uid
        profile['username'] = uname
        profile_records.append(profile)

    actor_profiles = pd.DataFrame(profile_records)
    actor_profiles.to_parquet(PROFILES_CACHE, index=False)
    print(f'Perfiles generados: {len(actor_profiles)}')

# Mostrar perfiles
display_cols = [c for c in ['username', 'role', 'radicalization_level', 'confidence', 'summary']
                if c in actor_profiles.columns]
if display_cols:
    print('\nPerfiles de actores:')
    for _, row in actor_profiles[display_cols].iterrows():
        print(f'\n  [{row.get("username", "")}]')
        for col in display_cols[1:]:
            print(f'    {col}: {row.get(col, "")}')

Perfiles cargados desde caché: 32 actores

Perfiles de actores:

  [Александр Славрос]
    role: moderator
    radicalization_level: high
    confidence: 0.9
    summary: Александр Славрос appears to be a high-level moderator on an extremist forum, likely with strong ties to historical and contemporary fascist ideologies. His posts indicate a role in welcoming new members, enforcing rules regarding avatars and signatures, and directing users to the appropriate sections of the forum for introductions, technical support, and news related to fascism and nationalism. The ideological indicators suggest a high level of radicalization.

  [Pro Patria Mori]
    role: active_member
    radicalization_level: high
    confidence: 0.9
    summary: Pro Patria Mori is an active member of extremist forums, demonstrating a high level of ideological commitment to fascist and ethnonationalist ideologies. The user frequently references historical figures such as Mussolini and Primo de Rivera, expresses a

## 7. Burrows' Delta: estilometría

**Burrows' Delta** es el estándar en **atribución de autoría** — la técnica que usan los
académicos para determinar si dos textos fueron escritos por la misma persona.

**¿Cómo funciona?**
En lugar de analizar el tema o el vocabulario especializado, mide las **palabras función**:
preposiciones, artículos, conjunciones, pronombres. Estas palabras se usan de forma
inconsciente e idiosincrática — cada persona tiene un patrón único que no cambia aunque
escriba sobre temas distintos.

El Delta entre dos usuarios es la diferencia media en z-score de esas palabras función.
**Delta bajo = estilo similar = posible misma persona (sockpuppet)**.

**Limitación importante:** un corpus homogéneo (todos los usuarios hablan sobre los mismos
temas extremistas) reduce el poder discriminante. Los candidatos son *leads*, no pruebas.

Este notebook reutiliza `_FUNCTION_WORDS` de `src/stylometry.py` como vocabulario de Delta. El resto de la implementación (z-score, matriz de distancias) es específica de este notebook — `src/stylometry.py` en sí implementa un método distinto (`extract_features`/`compare_users`, cosine similarity de 8 features), no Burrows' Delta.

In [17]:
# Preparar corpus para Burrows' Delta
# Mínimo 500 palabras por usuario — menos texto = Delta estadísticamente poco fiable
MIN_WORDS = 500

def clean_for_delta(text: str) -> str:
    """Limpiar texto para Delta: quitar HTML, URLs, y dejar solo letras en minúsculas."""
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return text.lower()

records = []
for uid, group in posts_clean.groupby('userid'):
    # uid_to_name.get(uid) sin segundo argumento: da None si uid no está en el dict,
    # a diferencia de .get(uid, uid) (que devolvía el propio uid como "nombre" y
    # nunca dejaba que pd.isna() detectara que no había usuario real).
    uname = uid_to_name.get(uid)
    if uname is None or pd.isna(uname): continue
    combined = ' '.join(group[text_col].dropna().astype(str).tolist())
    words    = clean_for_delta(combined).split()
    if len(words) >= MIN_WORDS:
        records.append({'userid': uid, 'username': str(uname), 'words': words, 'n_words': len(words)})

delta_corpus = pd.DataFrame(records).sort_values('n_words', ascending=False).reset_index(drop=True)
print(f'Usuarios con ≥{MIN_WORDS} palabras para Delta: {len(delta_corpus):,}')
print(f'Mediana de palabras por usuario: {delta_corpus["n_words"].median():,.0f}')
print(f'\nTop 10 usuarios con más texto:')
print(delta_corpus[['username', 'n_words']].head(10).to_string(index=False))

Usuarios con ≥500 palabras para Delta: 923
Mediana de palabras por usuario: 2,928

Top 10 usuarios con más texto:
         username  n_words
Александр Славрос  1275957
     Daddy Terror   752717
     Clive Bissel   599660
 Владимир_Борисов   518871
      Fascism=Fun   323798
              NSP   319815
       Myrrysmies   260244
         Der Löwe   237577
  Moorish Fascist   215360
         MOONLORD   208742


In [18]:
# Calcular Burrows' Delta — palabras función más frecuentes, top 150 usuarios
from src.stylometry import _FUNCTION_WORDS

N_FEATURES = 200  # Tope de palabras función a usar como features
TOP_USERS  = 150  # Analizar los 150 usuarios con más texto

# Obtener vocabulario: SOLO palabras función (_FUNCTION_WORDS), ordenadas por frecuencia.
# Burrows' Delta mide estilo, no tema — si dejamos pasar palabras de contenido
# (ideológicas, nombres propios) el Delta deja de medir estilo y empieza a medir de qué
# habla cada usuario, que es justo lo que NO queremos aquí.
all_words = [w for row in delta_corpus['words'] for w in row]
vocab     = [w for w, _ in Counter(all_words).most_common() if w in _FUNCTION_WORDS][:N_FEATURES]
vocab_set = set(vocab)
print(f'Vocabulario filtrado a palabras función: {len(vocab)} palabras (de un máximo de {N_FEATURES})')
print(f'vocab[:20]: {vocab[:20]}')

top_users_delta = delta_corpus.head(TOP_USERS).copy()

# Construir matriz de frecuencias relativas
# freq_matrix[i][j] = frecuencia de la palabra j en el corpus del usuario i
freq_matrix = np.zeros((len(top_users_delta), len(vocab)))
for i, (_, row) in enumerate(top_users_delta.iterrows()):
    counts = Counter(w for w in row['words'] if w in vocab_set)
    total  = row['n_words']
    for j, w in enumerate(vocab):
        freq_matrix[i, j] = counts.get(w, 0) / total

# Normalización por z-score por columna (por palabra)
# Cada palabra queda en la misma escala — palabras raras no dominan el resultado
mu_v    = freq_matrix.mean(axis=0)
sigma_v = freq_matrix.std(axis=0)
sigma_v[sigma_v == 0] = 1  # Evitar división por cero
z_matrix = (freq_matrix - mu_v) / sigma_v

# Delta = promedio de |z_i - z_j| para todos los features
n = len(top_users_delta)
delta_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = np.mean(np.abs(z_matrix[i] - z_matrix[j]))
        delta_matrix[i, j] = d
        delta_matrix[j, i] = d

delta_df_matrix = pd.DataFrame(
    delta_matrix,
    index=top_users_delta['username'].tolist(),
    columns=top_users_delta['username'].tolist(),
)

vals = delta_matrix[delta_matrix > 0]
print(f'\nMatriz Delta: {n}×{n}')
print(f'Delta medio: {vals.mean():.4f}  |  mínimo: {vals.min():.4f}  |  máximo: {vals.max():.4f}')

Vocabulario filtrado a palabras función: 176 palabras (de un máximo de 200)
vocab[:20]: ['the', 'of', 'and', 'to', 'a', 'in', 'that', 'it', 'you', 'for', 'as', 'be', 'not', 'this', 'they', 'with', 'on', 'have', 'but', 'we']



Matriz Delta: 150×150
Delta medio: 0.7747  |  mínimo: 0.2914  |  máximo: 4.9153


In [19]:
# Heatmap interactivo: top 40 usuarios por Burrows' Delta
# Verde oscuro = Delta bajo = estilos similares = candidatos a misma persona
TOP_HEAT = 40
sub    = delta_df_matrix.iloc[:TOP_HEAT, :TOP_HEAT]
labels = sub.index.tolist()

fig = go.Figure(go.Heatmap(
    z=sub.values,
    x=labels, y=labels,
    colorscale='RdYlGn',
    reversescale=True,  # Verde = similar (bajo Delta)
    hovertemplate='%{y} vs %{x}<br>Delta=%{z:.4f}<extra></extra>',
    colorbar=dict(title='Burrows Delta<br>(↓ = más similar)'),
))
fig.update_layout(
    title="Burrows' Delta — top 40 usuarios IronMarch<br>"
          "<sup>Verde oscuro = estilo similar → candidato a misma persona (sockpuppet)</sup>",
    template='plotly_dark', height=720, width=760,
    xaxis=dict(tickangle=-45, tickfont=dict(size=8)),
    yaxis=dict(tickfont=dict(size=8)),
    margin=dict(l=140, r=20, t=80, b=140),
)
fig.show()

In [20]:
# Candidatos a sockpuppet: pares con menor Delta
users_list = delta_df_matrix.index.tolist()
n = len(users_list)
pairs = []
name_to_nwords = dict(zip(delta_corpus['username'], delta_corpus['n_words']))

for i in range(n):
    for j in range(i + 1, n):
        pairs.append({
            'user_a':  users_list[i],
            'user_b':  users_list[j],
            'delta':   round(delta_df_matrix.iloc[i, j], 4),
            'words_a': name_to_nwords.get(users_list[i], 0),
            'words_b': name_to_nwords.get(users_list[j], 0),
        })

pairs_df = pd.DataFrame(pairs).sort_values('delta')
q01 = pairs_df['delta'].quantile(0.01)
q10 = pairs_df['delta'].quantile(0.10)

print('Top 20 pares con menor Delta (candidatos a sockpuppet):')
print(pairs_df.head(20)[['user_a', 'user_b', 'delta']].to_string(index=False))
print(f'\nPercentil  1%: Delta < {q01:.4f}')
print(f'Percentil 10%: Delta < {q10:.4f}')

suspicious = pairs_df[pairs_df['delta'] <= q01]
print(f'\nTop candidatos (percentil 1%, {len(suspicious)} pares):')
print(suspicious[['user_a', 'user_b', 'delta']].to_string(index=False))

# Persistir para que 04_sintesis_informe.ipynb lea el resultado real en vez de un
# valor hardcodeado (ver notebook 04, respuesta a P2)
pairs_df.to_parquet(IM_RESULTS / 'delta_pairs.parquet', index=False)
print(f'\ndelta_pairs.parquet guardado: {len(pairs_df):,} pares')

Top 20 pares con menor Delta (candidatos a sockpuppet):
           user_a            user_b  delta
          Rintrah        thecolonel 0.2914
Александр Славрос      Daddy Terror 0.2977
     Clive Bissel Ethno Nationalist 0.3038
  AmericanFascist       ProperPolak 0.3160
     Daddy Terror  Владимир_Борисов 0.3231
          Rintrah            Raycis 0.3428
 Владимир_Борисов            Aquila 0.3450
     Daddy Terror     Kill The Poor 0.3451
           Raycis        thecolonel 0.3515
 Владимир_Борисов             Havok 0.3525
 Владимир_Борисов            Mierce 0.3527
     Daddy Terror           Nowhere 0.3558
     Daddy Terror        thecolonel 0.3626
     Daddy Terror      Clive Bissel 0.3641
     Daddy Terror            Raycis 0.3643
       Myrrysmies             Raven 0.3644
       Myrrysmies            Raycis 0.3649
         MOONLORD            Cannon 0.3683
     Daddy Terror           Rintrah 0.3733
Александр Славрос     Kill The Poor 0.3747

Percentil  1%: Delta < 0.4294
Percentil 

### Cosine Delta: una segunda opinión sobre el mismo espacio de estilo

Burrows' Delta mide la distancia media absoluta entre los z-scores de palabras función
(una especie de distancia Manhattan). **Cosine Delta** usa el mismo vector z-scored de
palabras función, pero cambia la fórmula de distancia por **1 − similitud coseno**. No es
una técnica distinta —usa exactamente las mismas 176 palabras función y la misma
normalización—, es una **fórmula de distancia alternativa sobre el mismo espacio**.

Al ser el mismo `z_matrix` ya calculado, el coste computacional es prácticamente cero: una
sola llamada a `cosine_similarity`, sin volver a tocar Ollama ni reprocesar texto.

In [21]:
# Cosine Delta: misma matriz z-scored de Burrows' Delta, distancia coseno en vez de L1
from sklearn.metrics.pairwise import cosine_similarity

cos_sim = cosine_similarity(z_matrix)
cosine_delta_matrix = 1 - cos_sim  # 1 - similitud -> mismo sentido que Delta (bajo = parecido)

cosine_pairs = []
for i in range(n):
    for j in range(i + 1, n):
        cosine_pairs.append({
            'user_a': users_list[i], 'user_b': users_list[j],
            'cosine_delta': round(float(cosine_delta_matrix[i, j]), 4),
        })
cosine_pairs_df = pd.DataFrame(cosine_pairs)

# Cruzar con los pares de Burrows' Delta ya calculados (mismos 150 usuarios, mismos pares)
comparison = pairs_df.merge(cosine_pairs_df, on=['user_a', 'user_b'])

rho, p = spearmanr(comparison['delta'], comparison['cosine_delta'])
print(f"Correlación de Spearman Burrows Delta vs Cosine Delta: ρ={rho:.4f}  (n={len(comparison):,} pares)")

# Overlap de pares sospechosos (percentil 1%) según cada métrica
q01_burrows = comparison['delta'].quantile(0.01)
q01_cosine  = comparison['cosine_delta'].quantile(0.01)
susp_burrows = set(zip(
    comparison.loc[comparison['delta'] <= q01_burrows, 'user_a'],
    comparison.loc[comparison['delta'] <= q01_burrows, 'user_b'],
))
susp_cosine = set(zip(
    comparison.loc[comparison['cosine_delta'] <= q01_cosine, 'user_a'],
    comparison.loc[comparison['cosine_delta'] <= q01_cosine, 'user_b'],
))
overlap = susp_burrows & susp_cosine

print(f'\nPares sospechosos (percentil 1%) según Burrows Delta: {len(susp_burrows)}')
print(f'Pares sospechosos (percentil 1%) según Cosine Delta:  {len(susp_cosine)}')
print(f'Coinciden en ambas métricas: {len(overlap)}')
if overlap:
    print('\nPares que ambas métricas señalan como top sospechosos:')
    for a, b in sorted(overlap):
        row = comparison[(comparison['user_a'] == a) & (comparison['user_b'] == b)].iloc[0]
        print(f'  {a:<20} {b:<20} Burrows={row["delta"]:.4f}  Cosine={row["cosine_delta"]:.4f}')

print('\nTop 10 pares por Cosine Delta:')
print(comparison.nsmallest(10, 'cosine_delta')[['user_a', 'user_b', 'delta', 'cosine_delta']].to_string(index=False))

# Persistir la columna adicional para que nb04 pueda usarla también
comparison.to_parquet(IM_RESULTS / 'delta_pairs.parquet', index=False)
print(f"\ndelta_pairs.parquet actualizado con columna 'cosine_delta': {len(comparison):,} pares")

Correlación de Spearman Burrows Delta vs Cosine Delta: ρ=0.5398  (n=11,175 pares)

Pares sospechosos (percentil 1%) según Burrows Delta: 112
Pares sospechosos (percentil 1%) según Cosine Delta:  112
Coinciden en ambas métricas: 17

Pares que ambas métricas señalan como top sospechosos:
  AmericanFascist      ProperPolak          Burrows=0.3160  Cosine=0.2960
  Clive Bissel         Ethno Nationalist    Burrows=0.3038  Cosine=0.3451
  Daddy Terror         Fascism=Fun          Burrows=0.4164  Cosine=0.3630
  Four Suited Jack     Elburzito            Burrows=0.4042  Cosine=0.5008
  Havok                Will to Power        Burrows=0.4185  Cosine=0.4353
  Kay Kay Kay          youthknives          Burrows=0.4043  Cosine=0.5088
  MOONLORD             Cannon               Burrows=0.3683  Cosine=0.3884
  MOONLORD             Neizbezhnost         Burrows=0.3867  Cosine=0.4813
  Neizbezhnost         Cannon               Burrows=0.3945  Cosine=0.4712
  Neizbezhnost         Invictus             Bur

In [22]:
# Heatmap interactivo: top 40 usuarios por Cosine Delta
# Misma paleta y orientación que el heatmap de Burrows' Delta — verde oscuro = estilos similares
cosine_delta_df_matrix = pd.DataFrame(
    cosine_delta_matrix,
    index=users_list,
    columns=users_list,
)
sub_cos = cosine_delta_df_matrix.iloc[:TOP_HEAT, :TOP_HEAT]

fig = go.Figure(go.Heatmap(
    z=sub_cos.values,
    x=labels, y=labels,
    colorscale='RdYlGn',
    reversescale=True,  # Verde = similar (bajo Cosine Delta)
    hovertemplate='%{y} vs %{x}<br>Cosine Delta=%{z:.4f}<extra></extra>',
    colorbar=dict(title='Cosine Delta<br>(↓ = más similar)'),
))
fig.update_layout(
    title="Cosine Delta — top 40 usuarios IronMarch<br>"
          "<sup>Mismo espacio de palabras función que Burrows' Delta, distancia coseno en vez de L1</sup>",
    template='plotly_dark', height=720, width=760,
    xaxis=dict(tickangle=-45, tickfont=dict(size=8)),
    yaxis=dict(tickfont=dict(size=8)),
    margin=dict(l=140, r=20, t=80, b=140),
)
fig.show()

### Interpretación: ¿corrobora Cosine Delta a Burrows' Delta?

**Correlación moderada, no redundancia:** ρ=0.54 entre ambas métricas sobre los mismos
11.175 pares — están relacionadas (miden el mismo espacio de palabras función) pero no son
intercambiables. De los 112 pares en el percentil 1% de sospecha según cada métrica, solo
**17 coinciden en ambas** — la mayoría de candidatos de una lista no aparecen en la otra.

**Los dos protagonistas de P2 sobreviven a las dos métricas:** tanto `Rintrah`+`thecolonel`
(el mínimo global de Burrows) como `Александр Славрос`+`Daddy Terror` están en el percentil
1% de ambas fórmulas de distancia — esto es un dato a favor de esos dos pares
específicamente, no un artefacto de una sola fórmula.

**Aviso de divergencia:** el par con menor Cosine Delta de todo el corpus (`LITOS` +
`*_* Lusitanian Nationalist`, 0.2105) tiene un Burrows Delta de **0.9942** — casi el máximo
del corpus, el signo contrario. Dos fórmulas de distancia sobre el mismo espacio pueden
discrepar radicalmente para un par concreto. Ninguna métrica de estilometría es autoridad
por sí sola — por eso el cruce de señales de la síntesis final (notebook 04) exige que
coincidan varias técnicas, no una sola cifra.

In [23]:
# Guardar todas las señales semánticas para el notebook de síntesis
# Tabla de señales: una fila por usuario con todas las métricas computadas

semantic_signals = users_clean[['userid', 'username_raw']].copy()
semantic_signals = semantic_signals.rename(columns={'username_raw': 'username'})

# Agregar cluster HDBSCAN
if 'clusters' in dir() and best_ids is not None:
    cluster_map = dict(zip(best_ids, clusters))
    semantic_signals['hdbscan_cluster'] = semantic_signals['userid'].map(cluster_map)

# Agregar top topic BERTopic por usuario
if 'topic' in corpus_df.columns:
    top_topic_per_user = (
        corpus_df[corpus_df['topic'] != -1]
        .groupby('userid')['topic']
        .agg(lambda x: x.value_counts().idxmax())
        .reset_index()
        .rename(columns={'topic': 'top_topic'})
    )
    semantic_signals = semantic_signals.merge(top_topic_per_user, on='userid', how='left')

# Agregar min_delta (cuán similar estilísticamente es al usuario más parecido)
if 'delta_df_matrix' in dir() and len(delta_df_matrix) > 0:
    username_to_uid = dict(zip(users_clean['username_raw'], users_clean['userid']))
    min_delta_map = {}
    for uname in delta_df_matrix.index:
        vals_row = delta_df_matrix.loc[uname]
        vals_row = vals_row[vals_row > 0]
        min_delta_map[uname] = vals_row.min() if len(vals_row) > 0 else np.nan
    semantic_signals['min_delta'] = semantic_signals['username'].map(min_delta_map)

out_path = IM_RESULTS / 'semantic_signals.parquet'
semantic_signals.to_parquet(out_path, index=False)
print(f'Señales semánticas guardadas: {out_path}')
print(f'  {len(semantic_signals):,} usuarios')
print(f'  Columnas: {list(semantic_signals.columns)}')
print(f'\nSiguiente paso: 04_sintesis_informe.ipynb')

Señales semánticas guardadas: /home/miguel/csbc26/results/ironmarch/semantic_signals.parquet
  1,207 usuarios
  Columnas: ['userid', 'username', 'hdbscan_cluster', 'top_topic', 'min_delta']

Siguiente paso: 04_sintesis_informe.ipynb
